# Prepare the Embedding Shards

| Item | Description |
|---|---|
| Goal | Split the sentence table into contiguous shards so that a SLURM array can embed them in parallel (T3.6.3), and write the manifest that T3.6.4 uses to put the pieces back in order. |
| Input | `<scratch>/T3.6 - Firm Sentences/Processed/sentences.parquet` from T3.6.1. |
| Output | `<scratch>/T3.6 - Firm Sentences/Interim/shards/shard_<k>.parquet`, one per shard, and `<scratch>/T3.6 - Firm Sentences/Interim/shards/manifest.csv`; a copy of the manifest in `output/T3.6.2/`. |
| Unit | A sentence; shards are contiguous row ranges of the sentence table. |
| Rules | Eight shards of equal size in table order. Each shard row carries `row`, its position in the sentence table, so the embeddings can be aligned without re-reading the text. |
| Variables | • `row`: position of the sentence in `sentences.parquet`, from 0.<br>• `shard_id`: the shard a row belongs to, from 0 to 7. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and the shard count

Shards and the manifest live on scratch beside the sentence table; the manifest is also copied beside the notebook outputs.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.6.2"  # the identifier names this notebook's output folder
SENTENCES_FILE = SCRATCH_FOLDER / "T3.6 - Firm Sentences" / "Processed" / "sentences.parquet"
SHARDS_FOLDER = SCRATCH_FOLDER / "T3.6 - Firm Sentences" / "Interim" / "shards"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

N_SHARDS = 8  # one SLURM array task per shard
SHARDS_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Shards: {SHARDS_FOLDER}")
print(f"Outputs: {OUTPUT_FOLDER}")


Shards: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.6 - Firm Sentences/Interim/shards
Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.2


## Step 3. Load the sentence table and number its rows

The row number is the key that aligns the embedding matrix with the text later.


In [3]:
sentences = pd.read_parquet(SENTENCES_FILE)
assert not sentences.duplicated(["accessionNo", "sent_idx"]).any()
sentences["row"] = np.arange(len(sentences))
print(f"Sentences: {len(sentences):,} from {sentences['accessionNo'].nunique():,} filings")
display(sentences.head(3))


Sentences: 5,647,505 from 18,779 filings


,accessionNo,gvkey,fyear,sent_idx,sentence,row
0,0000912057-99-011120,001013,1999,0,"ADC Telecommunications, Inc. (""ADC"") was incor...",0
1,0000912057-99-011120,001013,1999,1,"ADC offers a broad range of network equipment,...",1
2,0000912057-99-011120,001013,1999,2,"ADC's broadband, multiservice network solution...",2


## Step 4. Assign shards and write them

Contiguous row ranges of equal size. Each shard file holds the row number, the filing, the sentence index, and the text.


In [4]:
sentences["shard_id"] = (sentences["row"] * N_SHARDS // len(sentences)).astype(int)
manifest_rows = []
for shard_id, shard in sentences.groupby("shard_id", sort=True):
    shard_file = SHARDS_FOLDER / f"shard_{shard_id}.parquet"
    shard[["row", "accessionNo", "sent_idx", "sentence"]].to_parquet(shard_file, index=False)
    manifest_rows.append({
        "shard_id": int(shard_id), "n_rows": int(len(shard)),
        "row_start": int(shard["row"].min()), "row_end": int(shard["row"].max()),
        "file": shard_file.name,
    })
manifest = pd.DataFrame(manifest_rows)
assert manifest["n_rows"].sum() == len(sentences)
assert (manifest["row_start"].iloc[1:].to_numpy() == manifest["row_end"].iloc[:-1].to_numpy() + 1).all()
print(f"Shards written: {len(manifest)}; rows per shard: {manifest['n_rows'].min():,} to {manifest['n_rows'].max():,}")
display(manifest)


Shards written: 8; rows per shard: 705,938 to 705,939


,shard_id,n_rows,row_start,row_end,file
0,0,705939,0,705938,shard_0.parquet
1,1,705938,705939,1411876,shard_1.parquet
2,2,705938,1411877,2117814,shard_2.parquet
3,3,705938,2117815,2823752,shard_3.parquet
4,4,705938,2823753,3529690,shard_4.parquet
5,5,705938,3529691,4235628,shard_5.parquet
6,6,705938,4235629,4941566,shard_6.parquet
7,7,705938,4941567,5647504,shard_7.parquet


## Step 5. Save the manifest

The copy beside the notebook outputs survives a scratch purge and records what the array embedded.


In [5]:
manifest_file = SHARDS_FOLDER / "manifest.csv"
manifest.to_csv(manifest_file, index=False)
manifest.to_csv(OUTPUT_FOLDER / "manifest.csv", index=False)
print(f"Saved: {manifest_file}")
print(f"Saved: {OUTPUT_FOLDER / 'manifest.csv'}")


Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.6 - Firm Sentences/Interim/shards/manifest.csv
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.2/manifest.csv
